In [32]:
import pandas as pd
import sqlite3

In [33]:
def transaction_import():
    # -------------------------
    # READ EXCEL
    # -------------------------
    data = pd.read_excel("data/eko_transaction.xlsx", dtype={"Number": str})


    # -------------------------
    # CLEAN MATERIAL
    # -------------------------
    data["Material"] = (
        data["Material"]
        .astype(str)
        .str.replace(r'^\d+\s*', '', regex=True)
        .str.strip()
    )


    # -------------------------
    # CLEAN VEHICLE
    # -------------------------
    data["Name"] = (
        data["Name"]
        .astype(str)
        .str.replace(" ", "", regex=False)
        .str.strip()
    )


    # -------------------------
    # SELECT COLUMNS
    # -------------------------
    data = data[
        [
            "Plant",
            "Name",
            "Number",
            "Material",
            "Date",
            "Bill.qty",
            "Bill.qty.1",
            "FinPr",
            "Tot Amount",
            "Auth.time"
        ]
    ]


    # -------------------------
    # CLEAN CARD NUMBERS
    # -------------------------
    data["Number"] = (
        data["Number"]
        .astype(str)
        .str.replace(r"\D", "", regex=True)
    )

    data = data[data["Number"].str.len() > 0]


    # -------------------------
    # REMOVE NON TRANSACTIONS
    # -------------------------
    data["Bill.qty"] = pd.to_numeric(data["Bill.qty"], errors="coerce")
    data = data[data["Bill.qty"] > 0]


    # -------------------------
    # RENAME COLUMNS FOR DB
    # -------------------------
    data = data.rename(columns={
        "Plant": "plant",
        "Name": "name",
        "Number": "card_number",
        "Material": "material",
        "Date": "date",
        "Bill.qty": "bill_qty",
        "Bill.qty.1": "bill_qty2",
        "FinPr": "price",
        "Tot Amount": "amount",
        "Auth.time": "auth_time"
    })


    # -------------------------
    # CONNECT DATABASE
    # -------------------------
    conn = sqlite3.connect("../eko_cards.db")


    # -------------------------
    # WRITE TO DATABASE
    # -------------------------
    data.to_sql(
        "transactions",
        conn,
        if_exists="append",
        index=False
    )


    conn.close()

    print("Transactions imported successfully.")

In [34]:
def company_import():
    cards = pd.read_excel("data/cards.xlsx", dtype={"Number": str})

    cards = cards.rename(columns={
        "Company": "company",
        "Name": "vehicle",
        "Number": "card_number"
    })

    conn = sqlite3.connect("../eko_cards.db")

    cards.to_sql(
        "cards",
        conn,
        if_exists="replace",
        index=False
    )

    conn.close()

In [35]:
def merge_tables():
    conn = sqlite3.connect("../eko_cards.db")

    data = pd.read_sql("""
    SELECT t.*, c.company
    FROM transactions t
    LEFT JOIN cards c
    ON t.card_number = c.card_number
    """, conn)

    conn.close()

    return data

In [36]:
def generate_result(data):
    output_file = "result/eko_transactions.xlsx"

    with pd.ExcelWriter(output_file, engine="openpyxl") as writer:

        for company, df in data.groupby("company", dropna=False):

            company_name = "Unknown" if pd.isna(company) else str(company)

            sheet_name = company_name[:31]

            df.to_excel(writer, sheet_name=sheet_name, index=False)
    print('Output file eko_transactions.xlsx generated successful!')

In [37]:
def delete_transactions():
    conn = sqlite3.connect("../eko_cards.db")
    cursor = conn.cursor()

    cursor.execute("DELETE FROM transactions")

    conn.commit()
    conn.close()

In [38]:
def delete_company():
    conn = sqlite3.connect("../eko_cards.db")
    cursor = conn.cursor()

    cursor.execute("DELETE FROM cards")

    conn.commit()
    conn.close()

In [39]:
def execute_pipeline():
    delete_transactions()

    transaction_import()

    data = merge_tables()

    generate_result(data)